# DroneFuse - Step 0: probe the MMAUD dataset (no GPU needed)

What this notebook does, without downloading the 370 GB dataset:
1. Measures the machine Kaggle gives us: CPUs, RAM, free disk on every folder, GPU, package versions.
2. Lists every MMAUD file on the Hugging Face Hub with its size, and totals them per sequence and sensor.
3. Reads the **index** of every ROS bag (12 sensor recordings + 13 ground-truth/calibration bags) over HTTP
   byte ranges - about 1 MB per bag - to get every topic, its message type, count, duration and rate.
4. Downloads a few chunks of one V1 and one V2/V3 recording, decodes one real message of every topic,
   and estimates how many GB each sensor takes.
5. Captures ~5 s of each audio stream and checks that it decodes (MMAUD stores MP3 inside the bags).
6. Downloads a handful of the extracted V1 files (images, LiDAR, radar, ground truth), one V2/V3 WAV file and
   one background-noise array, and records their real shape and format.
7. Measures the download speed from the Hub, to plan the extraction step.

**Kaggle settings:** Accelerator = **None (CPU)** - this step uses none of the 30 GPU hours. Internet = **On**.
**Inputs to add:** your `dronefuse-code` dataset only.
**Safe for "Save & Run All":** every step catches its own errors and the notebook always finishes and saves
`probe_summary.json` and `probe_report.txt`. Downloads go to a scratch folder that is deleted at the end,
so only a few MB are saved with the notebook.
**Expected run time:** about 20-40 minutes, depending on download speed.

In [ ]:
import os
import sys
from pathlib import Path

# The DroneFuse code is attached as a Kaggle dataset. Find the `dronefuse`
# package wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    # os.walk with followlinks=True also enters inputs that Kaggle mounts as links.
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('rosbag_index.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The dronefuse-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> dronefuse-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if (package.parent / '__init__.py').exists() and package.parent.name == 'dronefuse':
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder, so the files sit directly in
    # the dataset with no `dronefuse/` folder around them. In that case, link the
    # folder that holds rosbag_index.py under the name `dronefuse`.
    flat = package
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'dronefuse').exists():
        os.symlink(flat.parent, code_root / 'dronefuse')

sys.path.insert(0, str(code_root))
import dronefuse
print('DroneFuse code loaded from', Path(dronefuse.__file__).parent)

In [ ]:
# `rosbags` decodes ROS messages without installing ROS. Kaggle does not ship it.
import importlib
import subprocess

for package, module in (('rosbags', 'rosbags'), ('huggingface_hub', 'huggingface_hub')):
    try:
        importlib.import_module(module)
    except ImportError:
        result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], capture_output=True, text=True)
        print(f'pip install {package}:', 'ok' if result.returncode == 0 else result.stderr[-500:])

# The MMAUD README asks for the plain HTTP download path instead of the Xet client.
os.environ['HF_HUB_DISABLE_XET'] = '1'

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
REPO_ID = 'Yuan-Shenghai/MMAUD'
OUTPUT = Path('/kaggle/working/dronefuse_outputs/step0_probe')
TIME_BUDGET_MINUTES = 90          # later steps are skipped (not failed) once this is used up

# Recordings whose messages are decoded and whose audio is captured (one easy V1, one hard V2/V3).
SAMPLE_BAGS = ['V1/rosbag/Mavic3.bag', 'V2_V3/rosbag/m30s2.bag']
MAX_SAMPLE_MB_PER_BAG = 300       # chunk downloads for message decoding, per bag
AUDIO_SECONDS = 5.0

# Extracted files to look at (first FILES_PER_FOLDER files of each folder).
SAMPLE_FOLDERS = [
    'V1/filesystem_format/Mavic3/image',
    'V1/filesystem_format/Mavic3/lidar_360',
    'V1/filesystem_format/Mavic3/livox_avia',
    'V1/filesystem_format/Mavic3/radar_enhance_pcl',
    'V1/filesystem_format/Mavic3/ground_truth',
    'V2_V3/audio/m30s2',
    'background_noise/np_data_align/b1',
]
FILES_PER_FOLDER = 2
SPEED_TEST_MB = 300

In [ ]:
import json
import shutil

from dronefuse import environment
from dronefuse.runner import StepRunner

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}


def save_summary():
    summary['steps'] = runner.steps
    with open(OUTPUT / 'probe_summary.json', 'w') as f:
        json.dump(summary, f, indent=1, default=str)


summary['environment'] = runner.run('measure the machine', environment.collect)
SCRATCH = environment.pick_scratch_folder(min_free_gb=5)
SCRATCH.mkdir(parents=True, exist_ok=True)
summary['scratch'] = str(SCRATCH)
print(json.dumps(summary['environment'], indent=1, default=str))
save_summary()

In [ ]:
# Every file on the Hub with its size (saved as a compressed CSV for the next steps).
from dronefuse import hf_inventory

files = runner.run('list every MMAUD file on the Hub', hf_inventory.list_files, REPO_ID) or []
if files:
    hf_inventory.save_inventory(files, OUTPUT / 'mmaud_inventory.csv.gz')
    totals = hf_inventory.folder_totals(files)
    summary['inventory_files'] = len(files)
    summary['inventory_gb'] = round(sum(s for _, s in files) / 1e9, 2)
    summary['inventory_totals'] = totals
    print(f"{len(files)} files, {summary['inventory_gb']} GB")
    for folder, n, gb in totals[:30]:
        print(f'{gb:9.3f} GB {n:7d} files  {folder}')
save_summary()

In [ ]:
# The index of every bag: topics, message types, counts, durations, rates (~1 MB fetched per bag).
import requests
from dronefuse.probe import bag_sources, probe_bag_index

session = requests.Session()
sources = bag_sources(files)
print(f'{len(sources)} bags found:', list(sources))
summary['bags'], open_bags = [], {}

for bag_path, parts in sources.items():
    result = runner.run(f'index of {bag_path}', probe_bag_index, REPO_ID, bag_path, parts, session)
    if result is None:
        summary['bags'].append({'bag': bag_path, 'error': runner.steps[-1]['message']})
        continue
    bag_summary, index, reader = result
    summary['bags'].append(bag_summary)
    if bag_path in SAMPLE_BAGS:
        open_bags[bag_path] = (index, reader)
    print(f"{bag_path}: {bag_summary['size_gb']} GB, {bag_summary['seconds']} s, {len(bag_summary['topics'])} topics")
    for t in bag_summary['topics']:
        print(f"    {t['topic']:40s} {t['type']:36s} {t['messages']:7d} msgs {t['rate_hz']} Hz")
save_summary()

In [ ]:
# Decode one real message of every topic and estimate each sensor's total size.
from dronefuse.probe import probe_bag_messages

summary['bag_messages'] = {}
for bag_path, (index, reader) in open_bags.items():
    detail = runner.run(f'decode sample messages of {bag_path}', probe_bag_messages, reader, index,
                        n_spread=4, max_chunk_mb=64, max_total_mb=MAX_SAMPLE_MB_PER_BAG)
    summary['bag_messages'][bag_path] = detail or {'error': runner.steps[-1]['message']}
    if detail:
        for t in detail['topics']:
            print(f"{t['topic']:40s} {t.get('mean_kb')} kB/msg  ~{t.get('estimated_total_gb')} GB")
            print('     ', str(t.get('first_message'))[:300])
save_summary()

In [ ]:
# ~5 s of every audio stream, joined and decoded (the files are small and kept with the notebook).
from dronefuse.probe import capture_audio

summary['audio_capture'] = {}
for bag_path, (index, reader) in open_bags.items():
    folder = OUTPUT / 'audio_samples' / Path(bag_path).stem
    detail = runner.run(f'capture audio from {bag_path}', capture_audio, reader, index, folder, AUDIO_SECONDS)
    summary['audio_capture'][bag_path] = detail or {'error': runner.steps[-1]['message']}
    if detail and 'topics' in detail:
        for t in detail['topics']:
            print(t)
save_summary()

In [ ]:
# A few extracted files: real image size, array shapes, audio format.
from dronefuse.probe import sample_files

(OUTPUT / 'thumbnails').mkdir(exist_ok=True)
samples = runner.run('download and inspect sample files', sample_files, REPO_ID, files, SAMPLE_FOLDERS,
                     FILES_PER_FOLDER, SCRATCH / 'files', OUTPUT / 'thumbnails')
summary['file_samples'] = samples or []
for s in summary['file_samples']:
    print(s)
save_summary()

In [ ]:
# Download speed from the Hub (the data is streamed and thrown away).
from dronefuse.remote import hf_url, measure_download_speed

speed_file = 'V1/rosbag/Mavic3.bag'
summary['download_speed'] = runner.run('measure download speed', measure_download_speed,
                                       hf_url(REPO_ID, speed_file), SPEED_TEST_MB, 2_000_000_000, session)
print(summary['download_speed'])
save_summary()

In [ ]:
# The report: printed here and saved as probe_report.txt.
from dronefuse.probe import format_report

save_summary()
report = format_report(summary)
(OUTPUT / 'probe_report.txt').write_text(report)
print(report)

In [ ]:
# Clean up the scratch folder so nothing large is left behind.
shutil.rmtree(SCRATCH, ignore_errors=True)
saved = sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file())
print(f'Saved with the notebook: {saved / 1e6:.1f} MB in {OUTPUT}')
print('Steps that did not succeed:', runner.failed() or 'none')

## What to send back

1. The printed output of the **report** cell (or download `dronefuse_outputs/step0_probe/probe_report.txt`).
2. `probe_summary.json` and `mmaud_inventory.csv.gz` from the same folder - they go into the project's `docs/`
   and are used to design Step 1 (extraction).
3. If any step shows `error`, the traceback printed under that cell.

Everything in `/kaggle/working/dronefuse_outputs` is kept when you click **Save Version** (Save & Run All).